# NyaySetu — fine-tune Qwen3-4B Instruct on Claude-distilled legal Q&A

**What this notebook does (one click):** fine-tunes **Qwen3-4B-Instruct-2507** (Apache 2.0)
with [Unsloth](https://github.com/unslothai/unsloth) + LoRA on the `train.jsonl` / `val.jsonl`
dataset exported by `scripts/distill/export_dataset.py`, then exports a **q4_K_M GGUF** named
`nyaysetu-legal-qwen3-4b-q4_k_m.gguf` that you serve locally with Ollama as the model
**`nyaysetu-legal`** (see `training/Modelfile`).

**How to run (no ML knowledge needed):**
1. Colab menu: `Runtime → Change runtime type → T4 GPU` (the free tier works).
2. `Runtime → Run all`.
3. When **Cell 2** pauses, drag-and-drop your `train.jsonl` **and** `val.jsonl` into the upload
   box — or set `HF_DATASET_REPO` in that cell to download them from a Hugging Face dataset repo.
4. Wait (roughly 30–90 min depending on dataset size), then follow the **last cell** to download
   the GGUF and create the Ollama model.

Nothing in this notebook talks to the NyaySetu backend or to Anthropic — the teacher's answers
are already baked into the dataset. Each record is
`{"messages": [{"role": "user", ...prompt...}, {"role": "assistant", ...the teacher's JSON...}]}`
and we train the student to reproduce **only** the assistant JSON.


In [ ]:
# ============================================================================
# CELL 1 — Install Unsloth + training dependencies
# ----------------------------------------------------------------------------
# Unsloth pulls in everything else we need (transformers, trl, peft, accelerate,
# bitsandbytes). First run takes ~2-5 minutes on Colab — that is normal.
# If Colab asks to "Restart runtime" after install, restart and Run-all again.
# ============================================================================
!pip install -q unsloth

# If the released wheel ever lags behind (e.g. a brand-new model isn't recognised),
# uncomment the next line to install the nightly straight from GitHub:
# !pip install -q --upgrade --no-deps "unsloth @ git+https://github.com/unslothai/unsloth.git"

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU! In Colab: Runtime -> Change runtime type -> T4 GPU, then Run all again."
)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================================
# CELL 2 — Get the dataset (train.jsonl + val.jsonl)
# ----------------------------------------------------------------------------
# Two ways to provide the files produced by scripts/distill/export_dataset.py:
#
#   A) EASIEST: leave HF_DATASET_REPO = "" below. This cell will pause and show
#      an upload box -> drag-and-drop BOTH train.jsonl and val.jsonl into it.
#
#   B) OPTIONAL: if you uploaded the two files to a Hugging Face *dataset* repo
#      (e.g. "your-username/nyaysetu-distill"), put its id in HF_DATASET_REPO
#      and they are downloaded automatically (no clicking needed).
# ============================================================================
HF_DATASET_REPO = ""   # e.g. "your-username/nyaysetu-distill"  — leave "" to drag-and-drop

import os, shutil

if HF_DATASET_REPO:
    # Option B: pull both files from the HF dataset repo.
    from huggingface_hub import hf_hub_download
    for fname in ("train.jsonl", "val.jsonl"):
        cached = hf_hub_download(repo_id=HF_DATASET_REPO, filename=fname, repo_type="dataset")
        shutil.copy(cached, fname)
        print("downloaded", fname, "from", HF_DATASET_REPO)
else:
    # Option A: drag-and-drop upload (only asks for whatever is missing).
    missing = [f for f in ("train.jsonl", "val.jsonl") if not os.path.exists(f)]
    if missing:
        print("Please drag-and-drop these file(s) into the box below:", ", ".join(missing))
        from google.colab import files   # noqa: E402  (Colab-only import, on purpose)
        files.upload()
    still_missing = [f for f in ("train.jsonl", "val.jsonl") if not os.path.exists(f)]
    assert not still_missing, f"Missing {still_missing} — re-run this cell and upload them."

# Load with HF datasets. Tolerate a tiny smoke dataset whose val.jsonl is empty:
# in that case we reuse one training example as "val" so later cells still work.
from datasets import load_dataset

data_files = {"train": "train.jsonl"}
has_val = os.path.getsize("val.jsonl") > 0
if has_val:
    data_files["val"] = "val.jsonl"
dataset = load_dataset("json", data_files=data_files)
if not has_val:
    print("WARNING: val.jsonl is empty — reusing 1 train example as val (smoke-test mode).")
    dataset["val"] = dataset["train"].select(range(min(1, len(dataset["train"]))))

print(dataset)
print("\n--- first training example (user prompt, truncated) ---")
print(dataset["train"][0]["messages"][0]["content"][:600])
print("\n--- its target (the teacher's JSON the student must learn) ---")
print(dataset["train"][0]["messages"][1]["content"][:600])


In [ ]:
# ============================================================================
# CELL 3 — Load the base model: Qwen3-4B **Instruct** (2507), 4-bit via Unsloth
# ----------------------------------------------------------------------------
# * "Instruct-2507" is the non-thinking instruct variant — right for JSON answers
#   (no <think> blocks to strip). Apache 2.0 licensed.
# * load_in_4bit=True -> fits comfortably in a free T4's 16 GB.
# * MAX_SEQ_LEN 8192 matches data/distill/dataset_stats.md — check the
#   "max full sequence" line there; if it exceeds 8192 raise this (VRAM permitting).
#
# NOTE: verify the exact repo id at runtime — Unsloth occasionally renames its
# pre-quantised uploads. If the primary id 404s, the except-branch falls back to
# the plain Unsloth mirror and quantises to 4-bit on the fly (same result, a bit
# slower to load). You can also browse https://huggingface.co/unsloth for the
# current "Qwen3-4B-Instruct-2507" ids.
# ============================================================================
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 8192
MODEL_ID = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"   # pre-quantised (fast download)
FALLBACK_MODEL_ID = "unsloth/Qwen3-4B-Instruct-2507"           # fallback: quantise on the fly

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        dtype=None,          # auto -> float16 on a T4
    )
    print("Loaded:", MODEL_ID)
except Exception as e:
    print(f"Primary repo id failed ({e}); falling back to {FALLBACK_MODEL_ID} ...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=FALLBACK_MODEL_ID,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        dtype=None,
    )
    print("Loaded:", FALLBACK_MODEL_ID)


In [ ]:
# ============================================================================
# CELL 4 — Attach LoRA adapters (what actually gets trained)
# ----------------------------------------------------------------------------
# LoRA = small trainable matrices bolted onto the frozen 4-bit base model.
# We train ~1-2% of the parameters, which is why this fits on a free T4.
#   r=16, alpha=16, dropout=0  — the distillation recipe for this project.
#   Target modules: every attention + MLP projection (q/k/v/o + gate/up/down).
# ============================================================================
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,                      # 0 is Unsloth's fast path
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",      # attention
        "gate_proj", "up_proj", "down_proj",          # MLP
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",  # big memory saver for 8192-token sequences
    random_state=3407,
)
model.print_trainable_parameters()


In [ ]:
# ============================================================================
# CELL 5 — Format the dataset with the model's own chat template
# ----------------------------------------------------------------------------
# Each record is {"messages": [user, assistant]}. We render it through the
# tokenizer's chat template so the text looks EXACTLY like what Qwen3 sees at
# inference time (ChatML):
#
#   <|im_start|>user\n<the retrieval prompt><|im_end|>\n
#   <|im_start|>assistant\n<the teacher's JSON><|im_end|>\n
#
# In CELL 6 we wrap the trainer with Unsloth's train_on_responses_only, which
# masks everything except the assistant turn — so the model is graded ONLY on
# reproducing the teacher's JSON, never on parroting the prompt/context.
# These two marker strings tell it where the user turn and assistant turn start:
# ============================================================================
INSTRUCTION_PART = "<|im_start|>user\n"       # everything after this is masked ...
RESPONSE_PART = "<|im_start|>assistant\n"     # ... until this: the part we LEARN


def to_text(example):
    # tokenize=False -> return the rendered string; the trainer tokenizes later.
    # add_generation_prompt=False because the assistant answer is already present.
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}


# remove_columns=["messages"] matters: some trl versions auto-detect a "messages"
# column and would re-apply their own template on top of ours.
train_ds = dataset["train"].map(to_text, remove_columns=["messages"])
val_ds = dataset["val"].map(to_text, remove_columns=["messages"])

print("train:", len(train_ds), "| val:", len(val_ds))
print("\n--- rendered training text (truncated) ---")
print(train_ds[0]["text"][:800])


In [ ]:
# ============================================================================
# CELL 6 — Train (SFTTrainer + train_on_responses_only)
# ----------------------------------------------------------------------------
# The distillation recipe:
#   epochs 2 | batch 2 x grad-accum 4 (= effective batch 8) | lr 2e-4 cosine
#   warmup_ratio 0.03 | fp16 on T4 | log every 10 steps | seed 3407
# Expect ~30-90 min on a T4 depending on dataset size. Loss should fall fast in
# the first epoch and flatten in the second — that is normal and good.
# ============================================================================
import inspect
import torch
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

cfg_kwargs = dict(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,      # 2 x 4 = effective batch size 8
    num_train_epochs=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=not torch.cuda.is_bf16_supported(),   # T4 has no bf16 -> resolves to fp16=True
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",                 # memory-efficient optimizer
    weight_decay=0.01,
    seed=3407,
    output_dir="outputs",
    report_to="none",                   # no wandb prompts on Run-all
)

# trl renamed some fields across versions — set whichever this install accepts,
# so Run-all keeps working regardless of the pinned trl version.
accepted = set(inspect.signature(SFTConfig.__init__).parameters)
if "dataset_text_field" in accepted:
    cfg_kwargs["dataset_text_field"] = "text"
if "max_length" in accepted:
    cfg_kwargs["max_length"] = MAX_SEQ_LEN          # newer trl
elif "max_seq_length" in accepted:
    cfg_kwargs["max_seq_length"] = MAX_SEQ_LEN      # older trl

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(**cfg_kwargs),
)

# THE IMPORTANT BIT: mask the loss so ONLY the assistant JSON is learned.
trainer = train_on_responses_only(
    trainer,
    instruction_part=INSTRUCTION_PART,
    response_part=RESPONSE_PART,
)

# Sanity print: decode only the tokens that carry loss (labels != -100).
# You should see ONLY the teacher's JSON here — no prompt, no context.
_row = trainer.train_dataset[0]
_kept = [t for t, l in zip(_row["input_ids"], _row["labels"]) if l != -100]
print("--- text the model is actually graded on (truncated) ---")
print(tokenizer.decode(_kept)[:500])
print("---------------------------------------------------------")

trainer_stats = trainer.train()
print(trainer_stats)


In [ ]:
# ============================================================================
# CELL 7 — Inference sanity check on a validation example
# ----------------------------------------------------------------------------
# Feed the fine-tuned model one *held-out* prompt (never trained on) and assert
# the output parses as JSON with the fields the NyaySetu backend expects.
# If this cell passes, the GGUF you export next will speak the backend's language.
# ============================================================================
import json
import torch

FastLanguageModel.for_inference(model)   # 2x faster generation mode

sample = dataset["val"][0]
user_prompt = sample["messages"][0]["content"]
expected = json.loads(sample["messages"][1]["content"])   # the teacher's answer

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": user_prompt}],
    tokenize=True,
    add_generation_prompt=True,   # ends with "<|im_start|>assistant\n" -> model answers
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    out = model.generate(
        input_ids=inputs,
        max_new_tokens=1500,
        do_sample=False,                        # greedy = deterministic JSON
        pad_token_id=tokenizer.eos_token_id,
    )
generated = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

# Extract the JSON object (tolerate any stray text around it) and parse it.
start, end = generated.find("{"), generated.rfind("}")
assert start != -1 and end > start, f"Model produced no JSON object:\n{generated[:800]}"
parsed = json.loads(generated[start:end + 1])   # raises if not valid JSON -> cell fails loudly
for key in ("answer", "law_reference", "confidence", "reasoning"):
    assert key in parsed, f"Output JSON is missing required key: {key!r}"

print("SANITY CHECK PASSED — output is valid backend-shaped JSON.\n")
print("Student law_reference :", parsed.get("law_reference"))
print("Teacher law_reference :", expected.get("law_reference"))
print("\n--- student output (truncated) ---")
print(json.dumps(parsed, indent=2, ensure_ascii=False)[:1200])


In [ ]:
# ============================================================================
# CELL 8 — Save: LoRA adapter, merged 16-bit model, and the q4_K_M GGUF
# ----------------------------------------------------------------------------
# Three artefacts:
#   1. LoRA adapter (tiny, ~100 MB)     -> resume/iterate later without retraining
#   2. Merged fp16 model                -> base + adapter fused (input for GGUF)
#   3. GGUF q4_K_M (~2.5 GB)            -> the file Ollama serves
# The GGUF step compiles llama.cpp on first run — 10-20 min is NORMAL. Be patient.
# ============================================================================
import glob, os, shutil

LORA_DIR = "nyaysetu-legal-qwen3-4b-lora"
MERGED_DIR = "nyaysetu-legal-qwen3-4b-merged"
GGUF_DIR = "nyaysetu-legal-qwen3-4b-gguf"
GGUF_NAME = "nyaysetu-legal-qwen3-4b-q4_k_m.gguf"    # contract name — Modelfile expects this

# 1) adapter only
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print("saved adapter ->", LORA_DIR)

# 2) merged 16-bit (base weights + LoRA fused)
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print("saved merged ->", MERGED_DIR)

# 3) GGUF, quantised to q4_K_M (best quality/size trade-off for 4B on CPU/consumer GPU)
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")

# Unsloth names the file itself (varies by version), so find it and copy it to
# the exact name the Ollama Modelfile expects.
candidates = (
    glob.glob(f"{GGUF_DIR}/**/*.gguf", recursive=True)
    + glob.glob(f"{MERGED_DIR}/**/*.gguf", recursive=True)
    + [p for p in glob.glob("*.gguf") if p != GGUF_NAME]
)
assert candidates, "No .gguf produced — scroll up for llama.cpp errors and re-run this cell."
q4 = [p for p in candidates if "q4_k_m" in os.path.basename(p).lower()]
src = q4[0] if q4 else max(candidates, key=os.path.getsize)
shutil.copy2(src, GGUF_NAME)
print(f"\nGGUF ready: {GGUF_NAME}  ({os.path.getsize(GGUF_NAME) / 1e9:.2f} GB)")
print("Download it via the Files sidebar (folder icon on the left), or push to HF in the next cell")
print("(recommended — browser downloads of 2.5 GB from Colab often stall).")


In [ ]:
# ============================================================================
# CELL 9 — OPTIONAL: push adapter + GGUF to the Hugging Face Hub
# ----------------------------------------------------------------------------
# Recommended over downloading 2.5 GB through the browser. To use:
#   1. Set HF_PUSH_REPO below (the repo is created if it doesn't exist).
#   2. Run the cell — it asks you to paste a WRITE token from
#      https://huggingface.co/settings/tokens
# Leave HF_PUSH_REPO = "" to skip (the cell then does nothing).
# ============================================================================
HF_PUSH_REPO = ""   # e.g. "your-username/nyaysetu-legal-qwen3-4b" — "" skips this cell

if HF_PUSH_REPO:
    from huggingface_hub import HfApi, login
    login()   # paste your WRITE token when prompted

    # a) the LoRA adapter (tiny — lets you or others resume fine-tuning)
    model.push_to_hub(f"{HF_PUSH_REPO}-lora")
    tokenizer.push_to_hub(f"{HF_PUSH_REPO}-lora")
    print("pushed adapter ->", f"{HF_PUSH_REPO}-lora")

    # b) the GGUF, under its exact contract name so the Modelfile works verbatim
    api = HfApi()
    api.create_repo(HF_PUSH_REPO, exist_ok=True)
    api.upload_file(path_or_fileobj=GGUF_NAME, path_in_repo=GGUF_NAME, repo_id=HF_PUSH_REPO)
    print("pushed GGUF ->", f"https://huggingface.co/{HF_PUSH_REPO}/blob/main/{GGUF_NAME}")
    print("\nOn your machine, download it with:")
    print(f'  hf download {HF_PUSH_REPO} {GGUF_NAME} --local-dir nyaysetu-backend/training')
else:
    print("HF_PUSH_REPO is empty — skipping the Hub upload (nothing wrong, just optional).")


## Next steps — serve the model locally with Ollama

1. **Get the GGUF onto your machine** and place it in `nyaysetu-backend/training/`
   next to the `Modelfile`:
   - via the Colab Files sidebar (download `nyaysetu-legal-qwen3-4b-q4_k_m.gguf`), or
   - if you pushed to the Hub in Cell 9:
     ```bash
     hf download <your-username>/nyaysetu-legal-qwen3-4b nyaysetu-legal-qwen3-4b-q4_k_m.gguf --local-dir nyaysetu-backend/training
     ```

2. **Create the Ollama model** (Ollama must be installed and running):
   ```bash
   cd nyaysetu-backend/training
   ollama create nyaysetu-legal -f Modelfile
   ollama run nyaysetu-legal "Return only JSON: {\"ok\": true}"   # quick smoke test
   ```

3. **Point the backend at it** — in `nyaysetu-backend/.env` set:
   ```
   LLM_PROVIDER=ollama
   OLLAMA_MODEL=nyaysetu-legal
   ```
   then restart the API (`uvicorn app.main:app`). The RAG pipeline now generates with
   your distilled model instead of Claude — retrieval, citation verification,
   confidence gating and all other trust rails stay exactly as they were.

4. **Verify quality** before trusting it:
   ```bash
   export PYTHONIOENCODING=utf-8
   .venv/Scripts/python.exe scripts/answer_eval.py     # golden-set eval
   ```
   Compare against the Claude-teacher numbers; investigate any regression before deploying.
